# Amazon Electronics Product & Customer Review Analytics

**Objective:** compare product/category review activity and rating quality, analyse historical trends, and segment reviewer activity.

**Important:** this dataset contains ratings/review records, not verified orders or sales. Review counts are not sales volume or revenue. This notebook uses the project pipeline so the same cleaning and metrics can be regenerated consistently.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src" / "analysis_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from analysis_pipeline import load_and_clean, build_outputs

DATA_PATH = ROOT / "data" / "electronics.csv"
OUT = ROOT / "outputs"
df = load_and_clean(DATA_PATH)
print(f"Rows: {len(df):,} | Products: {df.item_id.nunique():,} | Reviewers: {df.user_id.nunique():,}")
print(f"Dates: {df.timestamp.min().date()} to {df.timestamp.max().date()}")

## 1. Data quality

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct_values": df.nunique()
}).sort_values("missing_pct", ascending=False)
quality

In [ ]:
print("Exact duplicates after cleaning:", df.duplicated().sum())
print("Invalid ratings remaining:", (~df.rating.between(1, 5)).sum())
print("Unknown brand share:", round((df.brand == "Unknown brand").mean()*100, 2), "%")

## 2. KPI overview

In [ ]:
kpis = pd.read_csv(OUT / "kpis.csv")
kpis.T.rename(columns={0: "value"})

## 3. Rating distribution

In [ ]:
rating = pd.read_csv(OUT / "rating_distribution.csv")
ax = rating.plot(kind="bar", x="rating", y="review_count", legend=False, figsize=(8,4), title="Rating distribution")
ax.set_xlabel("Rating (1–5)"); ax.set_ylabel("Review records"); plt.tight_layout()

## 4. Category performance: volume and satisfaction

In [ ]:
category = pd.read_csv(OUT / "category_performance.csv")
category.sort_values("review_count", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(category.review_count, category.average_rating, s=category.unique_products.clip(lower=1)*2, alpha=.8)
for _, r in category.iterrows(): ax.annotate(r.category, (r.review_count, r.average_rating), fontsize=8)
ax.set_xlabel("Review records (not sales)"); ax.set_ylabel("Average rating"); ax.set_title("Category review activity vs average rating")
plt.tight_layout()

## 5. Brand performance with a data-quality caveat

In [ ]:
brand = pd.read_csv(OUT / "brand_performance.csv")
brand[brand.brand != "Unknown brand"].sort_values("review_count", ascending=False).head(15)

## 6. Product performance with a minimum review threshold

In [ ]:
product = pd.read_csv(OUT / "product_performance.csv")
product.query("review_count >= 50").sort_values(["average_rating", "review_count"], ascending=[False, False]).head(20)

## 7. Reviewer activity segments

In [ ]:
segments = pd.read_csv(OUT / "reviewer_segments_summary.csv")
segments

In [ ]:
ax = segments.plot(kind="bar", x="reviewer_segment", y="reviewers", legend=False, figsize=(9,4), title="Reviewer activity segments")
ax.set_xlabel(""); ax.set_ylabel("Unique reviewer IDs"); plt.xticks(rotation=15, ha="right"); plt.tight_layout()

## 8. Monthly trends

In [ ]:
monthly = pd.read_csv(OUT / "monthly_trends.csv")
monthly.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(monthly.review_month, monthly.review_count)
ax.set_title("Monthly review volume"); ax.set_xlabel("Month"); ax.set_ylabel("Review records")
# Reduce crowded x-axis labels
for i, label in enumerate(ax.get_xticklabels()):
    if i % 12 != 0: label.set_visible(False)
plt.tight_layout()

## 9. Business interpretation and limitations

Use the generated `docs/key_findings.md` as a starting point, then inspect each finding against the tables. Good recommendations should focus on **which categories/products deserve further investigation**, not assert causes the data cannot prove.

Do not claim sales, revenue, conversion, product-page engagement, or customer retention from this dataset. It has no order records, prices, session IDs, or page-view/cart events. Reviewer segments describe review frequency only.